# Labelled WABAD reference spectrograms

**Input:** WABAD WAV/FLAC audio and matching Raven/CSV annotations. **Processing:** validate times/frequencies, compute a spectrogram, overlay the expert boxes with label colours. **Output:** `ground_truth.png` and a label legend for each selected recording.

Install from the README first. This notebook uses cell 1 of the supplied spectrogram notebook's plotting approach. Reference plots retain the original audio sampling rate and use a 2048-sample FFT / 512-sample hop; prediction plots use the detection pipeline's prepared waveform. Reference labels are annotations, not predicted species.

The default selects the real ARD paper recording. Download its waveform and expert table through the planned study Zenodo release; see [DATA_RELEASE.md](../docs/DATA_RELEASE.md). The original author-supplied reference figure is already available below.

![Original labelled ARD reference figure](../examples/paper/ARD_20211027_072000/ground_truth.png)

In [ ]:
from pathlib import Path
from adaptive_windowing.config import load_config
from adaptive_windowing.pipeline import records_for, validate_record
from adaptive_windowing import visualize

REPO = Path.cwd()
if not (REPO / "pyproject.toml").exists(): REPO = REPO.parent
CONFIG_PATH = REPO / "configs/paper_example.json"  # Use configs/wabad.json for the full study.
MAX_RECORDINGS = 1  # CHANGE THIS to None to draw every recording.
cfg = load_config(CONFIG_PATH)
records = records_for(cfg)
if MAX_RECORDINGS is not None: records = records[:MAX_RECORDINGS]
visualize.FMAX_HZ = cfg["parameters"]["FMAX"]

## Draw and save the references

Missing annotation files are skipped with a message. A header-only table creates a plot with no reference boxes. Every time coordinate is in seconds relative to its audio file, and frequency is in Hz. Use the PNG filename printed below to find the result.

In [ ]:
from IPython.display import display, Image
for record in records:
    references, _ = validate_record(record, cfg["parameters"])
    if references is None:
        print("No references:", record["audio"].name)
        continue
    folder = Path(cfg["output_dir"]) / record["site"] / record["audio"].stem
    folder.mkdir(parents=True, exist_ok=True)
    rois = [(b["t0"], b["t1"], b["f0"], b["f1"], b["label"]) for b in references]
    image_path = folder / "ground_truth.png"
    visualize.plot_spectrogram_with_rois(record["audio"], rois, image_path, record["site"])
    print(image_path)
    display(Image(filename=str(image_path)))